# dscraft.vision quickstart

This notebook demonstrates `dscraft.vision`'s dense image pipeline (`SimpleImagePipeline`: decode via Pillow -> augment resize/flip -> dense tensor), a small `TinyCNN` classifier, and the `torch.export()` -> ONNX export path with correctness verification.

**Section 1** runs the pipeline/model/export flow on a small in-memory synthetic gradient image.

**Section 2** repeats the exact same flow on a *real* handwritten-digit image from scikit-learn's bundled `sklearn.datasets.load_digits()` dataset (no network access).

This mirrors `packages/dscraft/examples/vision/export_cnn_example.py` in notebook form.

Requires the `vision` and `dev` extras (the latter for `scikit-learn`, used only to supply the real bundled digit dataset):
```bash
pip install -e "packages/dscraft[vision,dev]"
```

In [1]:
import io
import logging
import tempfile
import warnings
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from dscraft.vision import (
    ModelConfig,
    PipelineConfig,
    SimpleImagePipeline,
    build_model,
    export_to_onnx,
    resolve_device,
    synthetic_classification_batch,
    verify_export,
)

# torch.onnx's exporter registration logger warns (at WARNING level, via
# stdlib logging, not warnings.warn) that a handful of torchvision-specific
# ops (nms/roi_align/roi_pool/deform_conv2d) aren't registered when
# torchvision isn't installed. TinyCNN never uses any of those ops, so this
# is a benign, irrelevant notice for this notebook -- silenced narrowly by
# raising just this logger's level, not global logging config.
logging.getLogger("torch.onnx").setLevel(logging.ERROR)

# torch.export's internal pytree handling triggers a known upstream
# FutureWarning (`isinstance(treespec, LeafSpec)` deprecated) from
# copyreg during tracing -- an upstream torch.export implementation detail
# unrelated to anything this notebook does. Suppressed narrowly by message,
# not by broadly ignoring all FutureWarnings.
warnings.filterwarnings("ignore", category=FutureWarning, message=".*LeafSpec.*")

IMAGE_SIZE = 32
NUM_CLASSES = 10

device = resolve_device()
print(f"Resolved device: {device}")

Resolved device: mps


## Section 1: pipeline + TinyCNN + torch.export -> ONNX on a synthetic image

In [2]:
def make_synthetic_image_bytes(size: int) -> bytes:
    """A small in-memory synthetic gradient-pattern PNG -- keeps this notebook hermetic."""
    array = np.zeros((size, size, 3), dtype=np.uint8)
    array[:, :, 0] = np.linspace(0, 255, size, dtype=np.uint8)[None, :]
    array[:, :, 1] = np.linspace(255, 0, size, dtype=np.uint8)[:, None]
    array[:, :, 2] = 100
    buf = io.BytesIO()
    Image.fromarray(array, mode="RGB").save(buf, format="PNG")
    return buf.getvalue()


pipeline_config = PipelineConfig(image_size=IMAGE_SIZE, horizontal_flip_prob=0.5, seed=7, device=device)
pipeline = SimpleImagePipeline(pipeline_config)
raw_bytes = make_synthetic_image_bytes(size=40)

dense_tensor = pipeline.run(raw_bytes)
print(
    f"Pipeline produced a dense tensor: shape={tuple(dense_tensor.shape)}, "
    f"dtype={dense_tensor.dtype}, "
    f"range=[{dense_tensor.min():.3f}, {dense_tensor.max():.3f}]"
)

model_config = ModelConfig(in_channels=dense_tensor.shape[0], image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
model = build_model(model_config, seed=0, device=device)
print(f"Built TinyCNN: {sum(p.numel() for p in model.parameters())} parameters")

trace_batch = dense_tensor.unsqueeze(0).repeat(2, 1, 1, 1)  # (2, C, H, W)

with tempfile.TemporaryDirectory() as tmp_dir:
    onnx_path = Path(tmp_dir) / "tiny_cnn.onnx"
    export_to_onnx(model, trace_batch, onnx_path)
    print(f"Exported ONNX model to: {onnx_path} ({onnx_path.stat().st_size} bytes)")

    verification_batch, _ = synthetic_classification_batch(model_config, batch_size=8, seed=123, device=device)
    result = verify_export(model, onnx_path, verification_batch, atol=1e-4, rtol=1e-3)

    print("\n=== Correctness check: PyTorch vs. ONNX Runtime ===")
    print(f"  matched:       {result.matched}")
    print(f"  max_abs_diff:  {result.max_abs_diff:.3e}")
    print(f"  max_rel_diff:  {result.max_rel_diff:.3e}")
    assert result.matched

Pipeline produced a dense tensor: shape=(3, 32, 32), dtype=torch.float32, range=[0.008, 0.992]
Built TinyCNN: 11642 parameters


[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/tmpclmr8xxn/tiny_cnn.onnx (57373 bytes)



=== Correctness check: PyTorch vs. ONNX Runtime ===
  matched:       True
  max_abs_diff:  6.054e-08
  max_rel_diff:  1.873e-04


## Section 2: the same flow on a real handwritten digit (sklearn.datasets.load_digits())

In [3]:
from sklearn.datasets import load_digits

digits = load_digits()
image_0_16 = digits.images[0]  # (8, 8) float64, values in [0, 16]
real_label = int(digits.target[0])

image_uint8 = np.clip(image_0_16 / 16.0 * 255.0, 0, 255).astype(np.uint8)
pil_image = Image.fromarray(image_uint8, mode="L")
buf = io.BytesIO()
pil_image.save(buf, format="PNG")
real_raw_bytes = buf.getvalue()

print(f"Loaded real handwritten digit (true label={real_label}) from sklearn.datasets.load_digits().")

real_pipeline = SimpleImagePipeline(
    PipelineConfig(image_size=IMAGE_SIZE, horizontal_flip_prob=0.5, seed=7, device=device)
)
real_dense_tensor = real_pipeline.run(real_raw_bytes)
print(
    f"Pipeline produced a dense tensor: shape={tuple(real_dense_tensor.shape)}, "
    f"dtype={real_dense_tensor.dtype}, "
    f"range=[{real_dense_tensor.min():.3f}, {real_dense_tensor.max():.3f}]"
)

real_model_config = ModelConfig(in_channels=real_dense_tensor.shape[0], image_size=IMAGE_SIZE, num_classes=NUM_CLASSES)
real_model = build_model(real_model_config, seed=0, device=device)

real_trace_batch = real_dense_tensor.unsqueeze(0).repeat(2, 1, 1, 1)

with tempfile.TemporaryDirectory() as tmp_dir:
    real_onnx_path = Path(tmp_dir) / "tiny_cnn_real_digit.onnx"
    export_to_onnx(real_model, real_trace_batch, real_onnx_path)
    print(f"Exported ONNX model to: {real_onnx_path} ({real_onnx_path.stat().st_size} bytes)")

    real_verification_batch = real_dense_tensor.unsqueeze(0)
    real_result = verify_export(real_model, real_onnx_path, real_verification_batch, atol=1e-4, rtol=1e-3)

    print("\n=== Correctness check (real digit image): PyTorch vs. ONNX Runtime ===")
    print(f"  matched:       {real_result.matched}")
    print(f"  max_abs_diff:  {real_result.max_abs_diff:.3e}")
    print(f"  max_rel_diff:  {real_result.max_rel_diff:.3e}")
    assert real_result.matched

Loaded real handwritten digit (true label=0) from sklearn.datasets.load_digits().
Pipeline produced a dense tensor: shape=(3, 32, 32), dtype=torch.float32, range=[0.000, 0.902]


[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/tmpoue9x93_/tiny_cnn_real_digit.onnx (57373 bytes)



=== Correctness check (real digit image): PyTorch vs. ONNX Runtime ===
  matched:       True
  max_abs_diff:  1.118e-08
  max_rel_diff:  4.532e-06


## Summary

This notebook ran `dscraft.vision`'s decode -> augment -> dense-tensor pipeline, built a `TinyCNN`, exported it via `torch.export()` -> ONNX, and verified correctness against the original PyTorch model -- first on a synthetic gradient image, then on a real handwritten-digit image from `sklearn.datasets.load_digits()`.

See `packages/dscraft/README.md`'s `## dscraft.vision` section for more detail.